# sPHENIX Software Training — Notes
## Module 1: Foundations  +  Reader Week 1

**Date:** 2026-06-10
**Coverage:** Interactive course Module 1 (lessons 1.1–1.8 + checkpoint quiz) and Course Reader Chapter 1 (Week 1: *The Cluster, the Environment, and Your First ROOT Plot*).

| Lesson | Topic |
|---|---|
| 1.1 | What is sPHENIX? |
| 1.2 | Detector Subsystems |
| 1.3 | Getting Connected to SDCC |
| 1.4 | Environment Setup |
| 1.5 | Linux Essentials |
| 1.6 | Git & GitHub Workflow |
| 1.7 | ROOT Fundamentals |
| 1.8 | C++ for sPHENIX |
| 1.q | Checkpoint Quiz |


---
## 1.1 What is sPHENIX?

**sPHENIX** is a high-energy nuclear physics experiment at **RHIC** (Relativistic Heavy Ion Collider), Brookhaven National Laboratory. It studies the **Quark-Gluon Plasma (QGP)** — the state of matter that filled the universe for ~10 microseconds after the Big Bang, when it was too hot for protons/neutrons to form. The "s" = "super": it reuses the PHENIX interaction region (PHENIX ran 2000–2016) but is effectively a new detector with a new physics program.

### Four flagship measurements

| Measurement | What it probes |
|---|---|
| **Jet substructure** | How quarks/gluons lose energy crossing the QGP ("jet quenching") — probes the plasma at different length scales |
| **Upsilon spectroscopy** | Y(1S), Y(2S), Y(3S) bottomonium states "melt" at different QGP temperatures (color screening) → a plasma **thermometer** |
| **Open & closed heavy flavor** | Charm/bottom quarks produced early traverse the QGP → probe plasma **viscosity** (transport) |
| **Photons, π⁰, η** | Neutral mesons + direct photons → full kinematic range of QGP radiative processes |

### Collision systems

- **Au+Au @ √s_NN = 200 GeV** — main QGP production environment
- **p+p @ 200 GeV** — the "vacuum" baseline (no QGP)
- **p+Au @ 200 GeV** — controls for cold nuclear matter effects

QGP effects are isolated by comparing across systems via the **nuclear modification factor R_AA**.

### Where I fit in
Pick a **physics working group (PWG)**: Jets, Heavy Flavor, Upsilon, or Photon/Pi0 — plus usually a *technical responsibility* on a subsystem. First months = becoming **technically competent** (this course); physics ideas come after.


---
## 1.2 Detector Subsystems

sPHENIX = concentric layers around the interaction region (IR). Inside → out: **MVTX → INTT → TPC → EMCal → iHCal → (solenoid) → oHCal**, plus forward detectors (MBD, sEPD, ZDC). Each subsystem has a directory in the `coresoftware` repo.

| Subsystem | Role | Software dir |
|---|---|---|
| **MVTX** (pixels) | Precision vertex tracking, 3 silicon pixel layers — resolves displaced vertices (heavy flavor) | `mvtx/` |
| **INTT** (strips) | 4 silicon strip layers bridging MVTX↔TPC; fast timing for pile-up rejection | `intt/` |
| **TPC** | Main tracker — gas volume, measures trajectories & momenta; huge data volume | `tpc/`, `tpccalib/` |
| **EMCal** | W/SciFi SPACAL calorimeter — photons, electrons, π⁰; ~25k towers | `CaloBase/`, `CaloReco/` |
| **iHCal** | Steel-scintillator calorimeter *inside* solenoid — start of hadronic showers | `CaloBase/` |
| **oHCal** | Steel-scintillator *outside* solenoid — absorbs hadronic shower remnants | `CaloBase/` |
| **MBD** | Forward quartz Cherenkov counters — trigger, vertex timing, centrality | `mbd/` |
| **sEPD** | Segmented scintillator tiles — event plane for flow | `epd/` |
| **ZDC** | Very-forward neutron calorimeter — centrality, luminosity | `zdcinfo/` |

### The software stack (you live in layers 3–4)

```
Layer 4  YOUR CODE        (SubsysReco modules, ROOT macros)
Layer 3  coresoftware     (calo/ tracking/ trigger/ jets/ calibrations/)
Layer 2  Fun4All          (event loop engine: node tree, module chains, I/O)
Layer 1  External libs    (ROOT, Geant4, HepMC, FastJet, Eigen, Pythia8)
Layer 0  OS / infra       (AlmaLinux 9, CVMFS, GPFS, HTCondor)
```

> **Mental model to lock in:** an *event* = snapshot of everything in one bunch crossing, across all subsystems, as raw signals. *Reconstruction* turns raw signals into **physics objects** (clusters, tracks, vertices, jets). *Your analysis* queries and filters those objects.


---
## 1.3 Getting Connected to SDCC

All work happens on the **SDCC** cluster at BNL — there is no local-only workflow. Reader's key mental model: **"many machines, one filesystem"** — every interactive node is interchangeable because your files, the software, and the data live on shared network filesystems (GPFS/Lustre) visible to every node (this is also why Condor workers can see your files in Week 7).

### SSH gateway chain (two hops)

```bash
ssh <username>@ssh.sdcc.bnl.gov     # 1) gateway
ssh sphnxuser.sdcc.bnl.gov          # 2) sPHENIX interactive node
```

One-step version via `~/.ssh/config` on the local machine:

```
Host bnl-gw
  HostName ssh.sdcc.bnl.gov
  User yourname
  ServerAliveInterval 60

Host sphnx
  HostName sphnxuser.sdcc.bnl.gov
  User yourname
  ProxyJump bnl-gw
  ServerAliveInterval 60
```

Then just `ssh sphnx`.

### Filesystem layout

| Path | What lives here | Backed up? |
|---|---|---|
| `/sphenix/user/$USER/` | **Home** — code, macros, personal builds | ✅ |
| `/sphenix/data/` | Shared data (read-mostly) | ✅ |
| `/sphenix/sim/` | Simulation DSTs, generator output | Partial |
| `/sphenix/tg/` | Group **scratch** — don't park important data | ❌ |
| `/sphenix/lustre01/sphnxpro/` | **Production output** — real calibrated DSTs (source of truth) | — |
| `/opt/sphenix/core/` | sPHENIX software install root | Managed |
| `/cvmfs/sphenix.sdcc.bnl.gov/` | CVMFS read-only release distribution | Managed |

> ⚠️ **Quota discipline:** home quota ≈ 100 GB–1 TB. ROOT files are huge — check `quota -s` regularly; move old output to `/sphenix/tg/` or delete it.

First-login sanity commands: `whoami`, `pwd`, `hostname`, `quota -s`, `ls /cvmfs/sphenix.sdcc.bnl.gov/`.


---
## 1.4 Environment Setup

### The one command you ALWAYS run (every login, before anything else)

```bash
source /opt/sphenix/core/bin/sphenix_setup.sh -n new
```

It sets ~a dozen environment variables so ROOT, Geant4, CMake, and all sPHENIX libraries find each other.

### Build flavors

| Flag | Selects | Use when |
|---|---|---|
| `-n new` | Latest nightly/weekly build | Active development (default) |
| `-n ana.NNN` | Pinned "analysis" release (e.g. `ana.464`) | Reproducible analyses — every paper pins one |
| `-n pro.NNN` | Production build that made the DSTs | Official production / reprocessing |

> ⚠️ **Match your build to your data.** DSTs made with `ana.464` → analyze with `ana.464`. Mixing releases causes ABI mismatches, missing nodes, or *silent wrong answers*.

### Local install (for your own compiled modules)

```bash
source /opt/sphenix/core/bin/sphenix_setup.sh -n new
export MYINSTALL=/sphenix/user/$USER/install
source /opt/sphenix/core/bin/setup_local.sh $MYINSTALL
```

`setup_local.sh` prepends `$MYINSTALL/lib` to `LD_LIBRARY_PATH` and extends `ROOT_INCLUDE_PATH` + CMake prefixes — **without it, Fun4All cannot find the `.so` you build**. Put all three lines in `~/.bashrc`.

### Environment variables worth knowing

| Variable | Purpose |
|---|---|
| `$OPT_SPHENIX` | Root of the install tree (`/opt/sphenix/core`) |
| `$OFFLINE_MAIN` | Release directory of the selected build — every sPHENIX include & lib is under it |
| `$ROOTSYS` | Where ROOT lives |
| `$G4_MAIN` | Geant4 installation |
| `$CALIBRATIONROOT` | Calibration constants (runtime) |
| `$MYINSTALL` | Your personal install dir (you set it) |
| `$LD_LIBRARY_PATH` | Dynamic linker search path for `.so` files |
| `$ROOT_INCLUDE_PATH` | Header search path for ROOT macro JIT |

> 💡 **The #1 new-student mistake:** something breaks → first check `echo $OFFLINE_MAIN`. Empty = you forgot to source.


---
## 1.5 Linux Essentials

| Category | Commands |
|---|---|
| Navigation | `cd`, `ls -lah`, `pwd`, `find`, `cd -` (previous dir) |
| File ops | `cp`, `mv`, `rm -i` (prompt first), `mkdir -p a/{b,c,d}` (brace expansion) |
| Viewing | `cat`, `less`, `head -50`, `tail -100`, `tail -f` (follow live) |
| Searching | `grep -r` (recursive), `-n` (line nums), `-l` (filenames only), `--include="*.h"` |
| Processes | `ps aux \| grep root`, `top`/`htop`, `kill <PID>`, `kill -9` |
| Long-running | `nohup cmd > log 2>&1 &`; **tmux**: `tmux new -s name` → `Ctrl-B D` detach → `tmux attach -t name` |
| Transfer | `scp file sphnx:/path/`, `rsync -avz` (resumable, skips unchanged — the pro's choice) |

Useful `find` patterns: `find . -name "*.cc"` · `find ~ -size +1G` · `find . -name "*.root" -mtime -1`

> 💡 Inside a repo, `git grep` beats `grep -r` — only searches tracked files.

### Shell script template (the standard sPHENIX wrapper shape)

```bash
#!/bin/bash
set -euo pipefail                  # exit on error / undefined var / pipe failure

source /opt/sphenix/core/bin/sphenix_setup.sh -n new
export MYINSTALL=/sphenix/user/$USER/install
source /opt/sphenix/core/bin/setup_local.sh $MYINSTALL

RUN=${1:-48080}                    # arg 1 with default
NEVENTS=${2:-1000}
OUTDIR=/sphenix/user/$USER/output/run${RUN}
mkdir -p "$OUTDIR"

for f in /sphenix/lustre01/sphnxpro/run${RUN}/DST_CALO_*.root; do
    echo "Processing $f"
    root -l -b -q "Fun4All_MyAnalysis.C($NEVENTS, \"$f\")"
done
```

`chmod +x script.sh && ./script.sh 48080 1000`


---
## 1.6 Git & GitHub Workflow

All code: GitHub org **`sPHENIX-Collaboration`**.

| Repo | Contents | Usage |
|---|---|---|
| `coresoftware` | Reconstruction + framework code | Read often; PRs **from a fork** |
| `macros` | Official Fun4All macros (sim, reco) | Start every project from here |
| `tutorials` | Hello-world examples | Reference while learning |
| `analysis` | User physics analysis modules | Where my module will live; branch directly + PR |

### The canonical daily loop

```bash
git checkout master && git pull            # always start fresh
git checkout -b feature/pi0-efficiency     # ALWAYS branch before work
# ...edit...
git status && git diff
git add MyAnalysis.cc MyAnalysis.h
git commit -m "Add photon pair invariant mass histogram"
git push -u origin feature/pi0-efficiency
# → open PR on github.com
```

> 🚫 **Never:** commit directly on `master`/`main` · `git push --force` to a shared branch · commit large binaries (`.root`, `.tar.gz`).

### PR flow
Push branch → "Compare & pull request" → clear title (`[CaloReco] Fix cluster chi2 cut`) → describe what & **why** → request subsystem reviewers → respond to review by pushing more commits to the same branch → CI green + approval → maintainer merges. **Fix red CI before asking for review.**

Handy: `git log --oneline -20` · `git checkout -- file` (discard local edits) · `git reset HEAD file` (unstage) · `git commit --amend` (unshared only) · `git stash` / `git stash pop` · sync branch: merge or rebase `master`.


---
## 1.7 ROOT Fundamentals

ROOT = CERN's C++ analysis framework: interpreter (Cling), histograms, fitting, plotting, and the columnar file format. **Every DST is a ROOT file.** A ROOT "macro" is just a C++ file the interpreter runs — no compile step.

### Three ways to run

```bash
root -l                                  # interactive, no splash
root -l myAnalysis.C                     # run macro interactively
root -l -b -q myAnalysis.C               # batch: no graphics, auto-quit (scripts/Condor)
root -l -b -q 'myAnalysis.C(1000, "file.root")'   # with args — note single quotes
```

### The 90% classes

| Class | Purpose |
|---|---|
| `TFile` | Read/write ROOT files |
| `TTree` | Columnar ntuple — rows = events, columns = variables |
| `TH1F`/`TH1D` | 1D histogram (most-used class) |
| `TH2F`/`TH2D` | 2D histograms |
| `TProfile` | mean(Y) vs X in X-bins (resolution plots) |
| `TCanvas` | Drawing surface |
| `TLorentzVector` | 4-momentum + invariant mass arithmetic |
| `TF1` | 1D function for fits |
| `TLegend`, `TGraph(Errors)` | Legends; x-y points with errors |

### TTree quick inspection

```cpp
TFile *f = new TFile("data.root");
TTree *t = (TTree*)f->Get("ntp_cluster");
t->Print();                                // list branches
t->GetEntries();                           // row count
t->Draw("e");                              // 1D
t->Draw("e", "pt>1.0 && abs(eta)<1.1");    // with cut
t->Draw("eta:phi", "", "colz");            // 2D heatmap — "y:x", NOT division
```

### The canonical analysis macro pattern (memorize — used 100×)

```cpp
void ptAnalysis(const char* infile = "data.root") {
    // 1. Open file, grab tree
    TFile *f = new TFile(infile);
    TTree *t = (TTree*)f->Get("ntp_cluster");

    // 2. Hook variables to branches
    float e, pt, eta, phi;
    t->SetBranchAddress("e",   &e);
    t->SetBranchAddress("pt",  &pt);
    t->SetBranchAddress("eta", &eta);
    t->SetBranchAddress("phi", &phi);

    // 3. Book histograms
    TH1F *hpt = new TH1F("hpt", ";p_{T} [GeV/c];dN/dp_{T}", 100, 0, 10);

    // 4. Loop
    Long64_t n = t->GetEntries();
    for (Long64_t i = 0; i < n; ++i) {
        t->GetEntry(i);
        if (std::abs(eta) > 1.1) continue;
        if (e < 0.3) continue;
        hpt->Fill(pt);
    }

    // 5. Draw & save
    TCanvas *c = new TCanvas("c", "", 800, 600);
    c->SetLogy();
    hpt->Draw();
    c->SaveAs("pt_distribution.pdf");
}
```

### TLorentzVector — the secret weapon

```cpp
TLorentzVector p1, p2;
p1.SetPtEtaPhiM(pt1, eta1, phi1, 0.0);   // photon: m = 0
p2.SetPtEtaPhiE(pt2, eta2, phi2, e2);    // or by energy
TLorentzVector pair = p1 + p2;
double mass    = pair.M();               // invariant mass  ← pi0 reco later!
double pair_pt = pair.Pt();
double angle   = p1.Angle(p2.Vect());    // opening angle
```


In [ ]:
# Runnable sanity check: invariant mass of two photons (numpy mirror of TLorentzVector)
# Two photons from a pi0 decay should reconstruct m ~ 0.135 GeV
import numpy as np

def four_vec(pt, eta, phi, m=0.0):
    px, py = pt*np.cos(phi), pt*np.sin(phi)
    pz = pt*np.sinh(eta)
    E  = np.sqrt(px**2 + py**2 + pz**2 + m**2)
    return np.array([E, px, py, pz])

p1 = four_vec(1.2, 0.30, 0.10)   # photon 1
p2 = four_vec(0.9, 0.42, 0.32)   # photon 2
pair = p1 + p2
mass = np.sqrt(max(pair[0]**2 - np.sum(pair[1:]**2), 0))
print(f"invariant mass = {mass:.4f} GeV")

---
## 1.8 C++ for sPHENIX

sPHENIX ≈ 1M lines of C++. No templates/metaprogramming mastery needed — but fluency in the subset below is mandatory.

### Classes & inheritance — every analysis module IS a class

```cpp
// Base (framework-provided)
class SubsysReco {
 public:
    virtual int Init(PHCompositeNode* topNode) { return 0; }
    virtual int process_event(PHCompositeNode* topNode) = 0;  // pure virtual
    virtual int End(PHCompositeNode* topNode) { return 0; }
};

// Derived (my code)
class MyAnalysis : public SubsysReco {
 public:
    MyAnalysis(const std::string& name = "MyAnalysis");
    int Init(PHCompositeNode* topNode) override;
    int process_event(PHCompositeNode* topNode) override;
    int End(PHCompositeNode* topNode) override;
 private:
    std::string m_outname;
    TFile* m_file = nullptr;
};
```

**Key idea:** `virtual` + `override` = polymorphism. Fun4All holds a `SubsysReco*`, but calling `process_event` runs *my* override.

### Pointers & references

```cpp
SvtxTrack* track = iter->second;   // raw pointer
if (!track) return;                // ALWAYS null-check!
float px = track->get_px();        // -> dereferences
```

> 🚫 **Null pointers = #1 segfault cause.** Every `findNode::getClass<T>` call can return `nullptr` — always check before dereferencing.

### STL containers seen daily

```cpp
std::vector<TLorentzVector> photons;            // ordered, indexable workhorse
std::map<int, SvtxTrack*> tracks;               // key→value (track maps!)
for (auto& [id, track] : tracks) { ... }        // structured bindings
std::pair<float,float> eta_phi(0.5, 1.2);
std::set<int> run_numbers;                      // unique, sorted
std::deque<std::vector<TLorentzVector>> buf;    // event-mixing buffers (Module 4)
```

### Header / source split

- `MyAnalysis.h` — declarations (*what the class is*)
- `MyAnalysis.cc` — definitions (*what methods do*)

```cpp
#ifndef MYANALYSIS_H          // include guard: prevents double-inclusion
#define MYANALYSIS_H
#include <fun4all/SubsysReco.h>
#include <string>

class PHCompositeNode;        // forward declarations instead of #include
class TFile;                  // → much faster compiles, fewer dependencies
class TH1F;

class MyAnalysis : public SubsysReco { /* ... */ };
#endif
```

### Modern C++ bits

`auto` (type deduction) · range-based `for` · structured bindings `auto& [k,v]` · `nullptr` (never `NULL`/`0`) · uniform init `std::vector<int> v{1,2,3}` · `const` correctness.


---
## Reader Week 1 — extra points not in the lessons

- **"Many machines, one filesystem"** — the correct cluster mental model. Node A's writes are instantly visible on node B (and to Condor workers).
- All software is pre-installed under `/opt/sphenix` + served via **CVMFS** — you never install or compile the stack, you just *source* to find it.
- ROOT's **Cling** interpreter runs macros with no compile step — ideal for small analysis scripts.
- `root -l -b -q` is the combo for automated runs (bash wrappers, Condor) — no graphics, quit when done.
- The Reader's "hello world": `hello_gauss.C` — fill `TH1F` with 100k Gaussian randoms, `Fit("gaus")`, `SaveAs("hello_gauss.pdf")`. If that produces a PDF, the environment is sound.

### Check-for-Understanding (Reader §1.6) — with answers

**Q1.** *What does `echo $OFFLINE_MAIN` print before and after sourcing `sphenix_setup.sh`?*
→ Before: an **empty line** (unset). After: the release directory of the chosen build (e.g. `/cvmfs/sphenix.sdcc.bnl.gov/.../new`).

**Q2.** *Source setup but forget `setup_local.sh` — will ROOT find your compiled library in `$MYINSTALL/lib`?*
→ **No.** `setup_local.sh` is what prepends `$MYINSTALL/lib` to `LD_LIBRARY_PATH` (and adds include paths). Without it the dynamic linker never looks there — your `.so` is invisible to Fun4All/ROOT.

**Q3.** *TTree `T` in `data.root`: one-liner to plot pT with |eta| < 1.1.*
```cpp
root -l data.root -e 'T->Draw("pt", "abs(eta)<1.1")'
// or inside ROOT:  T->Draw("pt", "abs(eta)<1.1");
```


---
## Module 1 Checkpoint Quiz — answers

| # | Question (short) | Answer |
|---|---|---|
| 1 | Which collider / species? | **B** — RHIC · Au+Au and p+p at 200 GeV |
| 2 | Env setup command? | **B** — `source /opt/sphenix/core/bin/sphenix_setup.sh -n new` |
| 3 | ROOT can't find `libMyAnalysis.so` — first check? | **B** — `echo $MYINSTALL` (is the environment sourced?) |
| 4 | Official production DSTs live in? | **C** — `/sphenix/lustre01/sphnxpro/` |
| 5 | Correct git workflow? | **B** — feature branch → commit → push → PR → review |
| 6 | 4-momentum class? | **B** — `TLorentzVector` |
| 7 | Why forward declarations in headers? | **B** — reduces compile time & header dependencies |

---
## Muscle-memory cheat sheet (Week 1 distilled)

```bash
# Login ritual
ssh sphnx
source /opt/sphenix/core/bin/sphenix_setup.sh -n new
export MYINSTALL=/sphenix/user/$USER/install
source /opt/sphenix/core/bin/setup_local.sh $MYINSTALL
echo $OFFLINE_MAIN          # sanity: non-empty?

# ROOT
root -l -b -q 'macro.C(1000, "in.root")'

# When stuck, in order:
# 1. echo $OFFLINE_MAIN   (sourced?)
# 2. echo $MYINSTALL      (local install wired?)
# 3. quota -s             (out of disk?)
```

**Next up:** Module 2 — Fun4All, the node tree, SubsysReco lifecycle, and building my first module.
